# Integration test triage
The Bluecore Integration Test's recent runs from GitHub (through the dashboard's live API): pass rate, slow runs, and which tests failed and when.

To reproduce a failure, run the failing test against your local stack (see the stack's Integration Testing docs), and use the other starters to check the services it touches.

In [ ]:
import pandas as pd
import requests
from bluecore import DASHBOARD_API
data = requests.get(f"{DASHBOARD_API}/github/ci/blue-core-lod/bluecore-stack/bluecore-integration-test.yml", timeout=60).json()
runs = pd.DataFrame([
    {"run": r["number"], "result": r["conclusion"], "trigger": r["event"], "branch": r["branch"], "started": r["startedAt"],
     "minutes": round((r.get("durationSec") or 0) / 60, 1), "passed": (r.get("tests") or {}).get("passed"), "failed": (r.get("tests") or {}).get("failed"),
     "failed steps": ", ".join(r.get("failedSteps") or []), "url": r["url"]}
    for r in data["runs"]
])
print(data["name"], "-", len(runs), "recent runs of", data["totalRuns"])
runs.head(15)

## Pass rate and duration

In [ ]:
done = runs[runs["result"].isin(["success", "failure"])]
print(f"Pass rate: {100 * (done['result'] == 'success').mean():.1f}% over {len(done)} finished runs")
ax = runs.sort_values("run").plot(x="run", y="minutes", figsize=(9, 3), title="Duration of each run (minutes)", legend=False)

## The tests that failed, most often first

In [ ]:
fails = [
    {"test": f.get("name") or f.get("test"), "run": r["number"], "started": r["startedAt"], "error": (f.get("message") or "")[:160]}
    for r in data["runs"]
    for f in ((r.get("tests") or {}).get("failures") or [])
]
fails = pd.DataFrame(fails)
if fails.empty:
    print("No failing tests in these runs.")
else:
    display(fails.groupby("test").agg(times=("run", "count"), last_run=("run", "max")).sort_values("times", ascending=False).head(15))
    display(fails.sort_values("run", ascending=False).head(10))